In [1]:
import json
from pathlib import Path
import pandas as pd
import re
from scripts import extract as ext

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [3]:
with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/temas_ejercicio.json", "r") as file:
    temas_ejercicio = json.load(file)

with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/quimica_keywords.json") as file:
    quimica_keywords = json.load(file)

In [4]:
def classify_exercise(exam_name: str, statement: str, keywords = quimica_keywords) -> tuple:
    # obtener el año del examen
    exam_year = int(exam_name.split("_")[0])
    exam_name = (" - ".join(exam_name.split("_")[1:])).capitalize()

    # seleccionar modelo de examen en función del año
    if exam_year < 2020:
        exam_model = "modelo_2010_2019"
        exercise_number_pattern = r"^\d{1}\.-"

    elif exam_year < 2024:
        exam_model = "modelo_2020_2023"
        exercise_number_pattern = r"^[ABC]\d{1}"
    else:
        exam_model = "modelo_2024+"
        exercise_number_pattern = r"^\d{1}?[AB]"

    # temas del ejercicio según número ejercicio y modelo

    # "modelo_2024+": {
    #     "tipo_numeración" : "1A, 1B, 2A, 2B, ..., 4B, 5",
    #     "1": ["configuracion_electronica", ...]
    #     "2": ["equilibrio_quimico", ...],

    exercise_topics_dict = temas_ejercicio[exam_model]

    # obtener el número del ejercicio
    exercise_number = re.match(exercise_number_pattern, statement)

    # si se encuentra el número de ejercicio
    if exercise_number:

        # extraer la clave del diccionario del modelo de examen
        if exam_year < 2020:  # 2010 - 2019 - key 1,2,3,...
            exercise_number_key = exercise_number[0][0]

            # las keys son: "1", "2,3,4", "5,6"
            # exercise_number_key necesita un ajuste
            # para que se encuentre la clave
            for key in exercise_topics_dict.keys():
                if exercise_number_key in key:
                    exercise_number_key = key
                    break
        
        elif 2020 <= exam_year <= 2023:  # 2020 - 2023 - key A, B, C
            exercise_number_key = exercise_number[0][0]

        else:  # 2024 + - key 1, 2, 3, 4A, 4B, 5
            exercise_number_key = exercise_number[0]
            if exercise_number[0] not in  ["4A", "4B"]:
                exercise_number_key = exercise_number[0][0]
            else:
                exercise_number_key = exercise_number[0]

        # limpiar el número de ejercicio
        exercise_number = re.sub(r"[\.-]", "", exercise_number[0])

        # ejercicios de formulación - NO CLASIFICAR

        # condiciones a cumplir para ser ejercicio de formulación
        formulation_pre_2020 = exam_year <= 2019 and exercise_number_key == "1"
        formulation_2020_2023 =  (2020 <= exam_year <= 2023) and exercise_number_key == "A"

        if formulation_pre_2020 or formulation_2020_2023:
            exercise_topic = "Formulación"
            exercise_types = ["Formulación"]

        elif exam_year >= 2024 and exercise_number_key == "4A":
            exercise_topic = "Formulación"
            exercise_types = ["Formulación orgánica"]

        else:

            ###### CLASIFICACIÓN #######

            # opciones de tema para el ejercicio suministrado
            exercise_topic_options = exercise_topics_dict[exercise_number_key]

            # recorro los posibles temas del ejercicio
            # e intento clasificar el tema del ejercicio
            exercise_topic = ""
            for topic in exercise_topic_options:
                
                # obtengo las palabras clave que identifican el tema
                # y los tipos de ejercicios del tema y sus respectivas keywords
                topic_identifying_keywords = keywords[topic]["keywords_tema"]
                for kw in topic_identifying_keywords:
                    if kw in statement:
                        exercise_topic = topic
                        break

                # si se ha clasificado el ejercicio deja de buscar
                if exercise_topic:
                    break

            # lista de tipos de ejercicio
            exercise_types = []

            # si se ha identificado el tema
            # lo siguiente es asignar tipo(s) de ejercicio
            if exercise_topic:
                # compruebo existencia de keywords de cada tipo de ejercicio en el enunciado
                exercise_type_keywords = keywords[exercise_topic]["tipos"]
                for t, kw_list in exercise_type_keywords.items():
                    # comprueba lista de keywords para ese tipo de ejercicio
                    # con que haga match una keyword con el enunciado
                    # clasifícalo com ese tipo de ejercicio
                    if any(kw in statement for kw in kw_list):
                        exercise_types.append(t)
            else:
                exercise_topic, exercise_types = None, [] 
    else:
        exercise_topic, exercise_types = None, []

    return (exam_year, exam_name, exercise_number, exercise_topic, exercise_types, statement)

In [5]:
def classify_exam_period(exam_year):
    if exam_year < 2020:
        exam_period = "2010-2019"
    elif exam_year <= 2024:
        exam_period = "2020-2024"
    else:
        exam_period = "2025+"

    return exam_period

## Test con un archivo

In [6]:
# file = ROOT / "examenes_quimica/2013_quimica/2013_quimica_sel_2013_quimica - junio - titular - 3-Examen-Andalucía.pdf"

In [7]:
# exam_text = ext.extract_exercises_from_pdf(file)

# exam_year = int(file.name.split("_")[0])
# exam_period = classify_exam_period(exam_year)

# exercises_list = ext.split_exercises(exam_text, exam_period=exam_period)

# unclassified_exercises_df = pd.DataFrame([{"year" : exam_year, "exam_option" : exam_option, "statement" : st} for exam_option, st in exercises_list])

In [8]:
# unclassified_exercises_df.head()

Enunciado de un ejercicio

In [9]:
# print(exercises_list[4][1])

Clasificar ejercicio

In [10]:
# classify_exercise(file.stem, exercises_list[11][1], quimica_keywords)

## Clasificar los ejercicios de un año

In [11]:
# classifications = []
# for file in (ROOT / "examenes_quimica/2025_quimica").iterdir():
#     if file.is_file():
#         exam_text = ext.extract_exercises_from_pdf(file)
#         exam_year = int(file.name.split("_")[0])

#         if exam_year < 2020:
#             exam_period = "2010-2019"
#         elif exam_year <= 2024:
#             exam_period = "2020-2024"
#         else:
#             exam_period = "2025+"

#         exercises_list = ext.split_exercises(exam_text, exam_period=exam_period)

#         # exercises_list devuelve una tupla: (exam_option, statement)
#         for exercise_tuple in exercises_list:
#             try:
#                 statement = exercise_tuple[1]
#                 exam_option = exercise_tuple[0]
#                 classification = classify_exercise(file.stem, statement, keywords=quimica_keywords)
#                 classification = list(classification)
#                 classification.append(exam_option)
#                 classifications.append(classification)
#             except Exception as e:
#                 print(f"Error: file {file.name}. Exception: {e}")

# df_year = pd.DataFrame(classifications, columns = ["year", "exam_name", "exercise_number", "topic", "exercise_type", "statement", "exam_option"])
# df_year = df_year.explode("exercise_type", ignore_index=True)

In [12]:

# df_year = pd.DataFrame(classifications, columns = ["year", "exam_name", "exercise_number", "topic", "exercise_type", "statement", "exam_option"])
# df_year = df_year.explode("exercise_type", ignore_index=True)

In [13]:
# df_year.head()

## Clasificar todos los exámenes

In [29]:
classifications = []
exam_files = sorted((ROOT / "examenes_quimica").rglob("*.pdf"))

for file in exam_files:
    try:
        if file.is_file():
            exam_text = ext.extract_exercises_from_pdf(file)
            exam_year = int(file.name.split("_")[0])

            exam_period = classify_exam_period(exam_year)

            exercises_list = ext.split_exercises(exam_text, exam_period=exam_period)

            # exercises_list devuelve una tupla: (exam_option, statement)
            for exercise_tuple in exercises_list:
                try:
                    statement = exercise_tuple[1].strip()
                    exam_option = exercise_tuple[0]
                    classification = classify_exercise(file.stem, statement, keywords=quimica_keywords)
                    classification = list(classification)
                    classification.append(exam_option)
                    classification.append(file.relative_to(ROOT))
                    
                    classifications.append(classification)
                except Exception as e:
                    print(f"Error: file {file.name}. Exception: {e}")
    except Exception as e:
        print(f"{file.name}. Error: {e}")

In [30]:

df_classified = pd.DataFrame(classifications, columns = ["year", "exam_name", "exercise_number", "topic", "exercise_type", "statement", "exam_option", "exam_file_path"])
df_classified = df_classified.explode("exercise_type", ignore_index=True)

In [16]:
df_classified.head()

,year,exam_name,exercise_number,topic,exercise_type,statement,exam_option,exam_file
0,2010,Quimica - quimica - quimica - junio - quimica ...,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A,2010_quimica_quimica - quimica_junio - quimica...
1,2010,Quimica - quimica - quimica - junio - quimica ...,2,enlace_químico,Comparación Born-Haber,2.- Supongamos que los sólidos cristalinos NaF...,Opción A,2010_quimica_quimica - quimica_junio - quimica...
2,2010,Quimica - quimica - quimica - junio - quimica ...,3,solubilidad,influencia del pH sobre la solubilidad,3.- Se dispone de una disolución acuosa satura...,Opción A,2010_quimica_quimica - quimica_junio - quimica...
3,2010,Quimica - quimica - quimica - junio - quimica ...,4,termoquímica,Cálculo de entropía,4.- Razone si las siguientes afirmaciones son ...,Opción A,2010_quimica_quimica - quimica_junio - quimica...
4,2010,Quimica - quimica - quimica - junio - quimica ...,4,termoquímica,verdadero o falso,4.- Razone si las siguientes afirmaciones son ...,Opción A,2010_quimica_quimica - quimica_junio - quimica...


In [17]:
# df_classified.isna().sum()

In [31]:
df_classified.to_csv("df_classified.csv", index=False, encoding="utf-8")

In [32]:
df_classified[df_classified.topic == "Formulación"].to_csv("assign_exam_names/form_classified.csv", index=False, encoding="utf-8")

In [20]:
df_classified.year.aggregate(["min", "max"])

min    2010
max    2026
Name: year, dtype: int64